In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from liffile import LifFile

from skimage import util, exposure
from skimage.io import imread
from skimage.measure import label, regionprops_table, regionprops,  moments_central
from skimage.morphology import  disk, remove_small_objects, binary_dilation, remove_small_holes , erosion, closing, skeletonize, binary_closing, disk, binary_opening, binary_erosion
from skimage.transform import rotate
from skimage.transform import probabilistic_hough_line
from skimage.filters import threshold_sauvola, frangi, sato, threshold_yen, threshold_local, threshold_mean, threshold_li, threshold_minimum, try_all_threshold, threshold_triangle, median, sobel, gaussian, threshold_otsu, sobel_v
from skimage.graph import route_through_array
from skimage.draw import line as sk_line

import napari
from matplotlib.colors import to_rgba

import warnings
warnings.filterwarnings("ignore")

def label_colormap(color, alpha=1.0):
    rgba = np.array(to_rgba(color))
    rgba[3] = alpha
    return {
    0: np.array([0., 0., 0., 0.]),
    1: rgba
    }


In [508]:
"""
Barrier line extraction + z-consistency filtering + segmentation.

Core outputs:
- per-slice path masks + y(x) profiles
- keep a consistent contiguous z-band (handles gaps + trims outlier ends)
- segment above/below the (optionally smoothed) barrier surface

Dependencies: numpy, scipy, scikit-image
"""

from __future__ import annotations

from dataclasses import dataclass
from heapq import heappop, heappush
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
from scipy.ndimage import gaussian_filter
from skimage import exposure
from skimage.draw import line as sk_line
from skimage.filters import median, sato, threshold_li, threshold_minimum, threshold_otsu, threshold_triangle
from skimage.measure import label, regionprops
from skimage.morphology import (
    binary_closing,
    binary_dilation,
    binary_erosion,
    binary_opening,
    disk,
    remove_small_holes,
    remove_small_objects,
    skeletonize,
)


# -------------------------
# Small utilities
# -------------------------

def vertical_brightness_ratio(img: np.ndarray, eps: float = 1e-6) -> float:
    """Abs(log(median(top)/median(bottom))). Used to gate edge vs ridge mode."""
    y = img.shape[0] // 2
    a = float(np.median(img[:y]))
    b = float(np.median(img[y:]))
    return float(abs(np.log((a + eps) / (b + eps))))


def fit_line_fast(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """Fast least-squares y=a*x+b, avoiding np.polyfit overhead."""
    n = x.size
    if n < 2:
        return 0.0, float(y[0]) if n else 0.0

    x = x.astype(np.float32, copy=False)
    y = y.astype(np.float32, copy=False)

    xm, ym = x.mean(), y.mean()
    dx = x - xm
    denom = float(np.dot(dx, dx))
    if denom == 0.0:
        return 0.0, float(ym)

    a = float(np.dot(dx, y - ym) / denom)
    b = float(ym - a * xm)
    return a, b


def extend_path_to_edges(path_mask: np.ndarray, k: int = 5, max_gap: int = 30, edge_tol: int = 0) -> np.ndarray:
    """
    Extend a path to x=0 and/or x=W-1 if its endpoints are within max_gap of those edges.
    Uses k extreme points and a linear fit to extrapolate.
    """
    m = path_mask.astype(bool).copy()
    H, W = m.shape
    ys, xs = np.nonzero(m)
    n = xs.size
    if n < 2:
        return m

    x_min, x_max = int(xs.min()), int(xs.max())
    left_gap, right_gap = x_min, (W - 1) - x_max

    if left_gap <= edge_tol and right_gap <= edge_tol:
        return m
    if left_gap > max_gap and right_gap > max_gap:
        return m

    kk = min(k, n)

    if left_gap <= max_gap:
        idx = np.argpartition(xs, kk - 1)[:kk]
        a, b = fit_line_fast(xs[idx], ys[idx])
        x0 = x_min
        y0 = int(np.clip(np.rint(a * x0 + b), 0, H - 1))
        y_edge = int(np.clip(np.rint(b), 0, H - 1))  # x=0
        rr, cc = sk_line(y_edge, 0, y0, x0)
        m[rr, cc] = True

    if right_gap <= max_gap:
        idx = np.argpartition(xs, n - kk)[-kk:]
        a, b = fit_line_fast(xs[idx], ys[idx])
        x1 = x_max
        y1 = int(np.clip(np.rint(a * x1 + b), 0, H - 1))
        y_edge = int(np.clip(np.rint(a * (W - 1) + b), 0, H - 1))
        rr, cc = sk_line(y1, x1, y_edge, W - 1)
        m[rr, cc] = True

    return m


def _largest_component(mask: np.ndarray) -> np.ndarray:
    lab = label(mask)
    if lab.max() == 0:
        return mask.astype(bool)
    areas = np.bincount(lab.ravel())
    areas[0] = 0
    keep = int(np.argmax(areas))
    return (lab == keep)


def _largest_n_components(mask: np.ndarray, n_keep: int = 3) -> np.ndarray:
    lab = label(mask)
    if lab.max() == 0:
        return mask.astype(bool)
    areas = np.bincount(lab.ravel())
    areas[0] = 0
    keep = np.argsort(areas)[-n_keep:]
    return np.isin(lab, keep)


def extract_boundary(
    img: np.ndarray,
    thr_func,
    min_size: int = 100,
    hole_area: int = 500,
    opening_size: int = 3,
) -> Tuple[np.ndarray, np.ndarray]:
    """Threshold + clean + keep largest region + return interface boundary as a 1px mask."""
    bw = img > thr_func(img)
    if min_size > 0:
        bw = remove_small_objects(bw, min_size=min_size)
    if hole_area > 0:
        bw = remove_small_holes(bw, area_threshold=hole_area)
    if opening_size > 0:
        bw = binary_opening(bw, disk(opening_size))

    bw = _largest_component(bw)

    # boundary of region
    path = bw ^ binary_erosion(bw, disk(1))
    path = _largest_component(path)
    return path.astype(bool), bw.astype(bool)


# -------------------------
# Routing on skeleton
# -------------------------

_NEIGH8 = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]


def find_best_left_right_route(mask: np.ndarray, edge_margin: int = 30, step_cost: float = 100.0) -> Optional[np.ndarray]:
    """
    Multi-source Dijkstra from any left-edge point to any right-edge point.
    Prefers paths on mask; allows small off-mask moves via penalty.
    """
    H, W = mask.shape
    left = np.argwhere(mask[:, :edge_margin])
    right = np.argwhere(mask[:, W - edge_margin :])

    if left.size == 0 or right.size == 0:
        m2 = binary_dilation(mask, disk(2))
        left = np.argwhere(m2[:, :edge_margin])
        right = np.argwhere(m2[:, W - edge_margin :])
        if left.size == 0 or right.size == 0:
            return None
        mask = m2

    right_set = {(int(y), int(W - edge_margin + x)) for y, x in right}

    off_penalty = step_cost * 0.2
    cost = np.where(mask, step_cost, step_cost + off_penalty).astype(np.float32, copy=False)

    visited = np.zeros((H, W), dtype=bool)
    prev: Dict[Tuple[int, int], Optional[Tuple[int, int]]] = {}
    pq: List[Tuple[float, Tuple[int, int]]] = []

    for y, x in left:
        p = (int(y), int(x))
        heappush(pq, (0.0, p))
        prev[p] = None

    end: Optional[Tuple[int, int]] = None

    while pq:
        c, (y, x) = heappop(pq)
        if visited[y, x]:
            continue
        visited[y, x] = True
        if (y, x) in right_set:
            end = (y, x)
            break

        for dy, dx in _NEIGH8:
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W and not visited[ny, nx]:
                nc = c + float(cost[ny, nx])
                heappush(pq, (nc, (ny, nx)))
                prev.setdefault((ny, nx), (y, x))

    if end is None:
        return None

    out = np.zeros((H, W), dtype=bool)
    cur: Optional[Tuple[int, int]] = end
    while cur is not None:
        out[cur] = True
        cur = prev[cur]
    return out


# -------------------------
# Per-slice line extraction
# -------------------------

@dataclass(frozen=True)
class RidgeParams:
    ratio_cutoff: float = 0.75
    med_disk: int = 7

    # edge mode fallbacks
    edge_extend_gap: int = 30

    # ridge mode settings
    sato_sigmas: Tuple[float, ...] = (2, 3, 4, 5, 6, 7, 8)
    ridge_close_disk: int = 9
    ridge_min_size: int = 600
    ridge_keep_components: int = 3
    ridge_extend_gap: int = 50
    edge_margin: int = 30
    step_cost: float = 100.0


def find_ridge_through_plane(img: np.ndarray, params: RidgeParams = RidgeParams()) -> Tuple[Optional[np.ndarray], str, float]:
    """
    Returns (path_mask, mode, brightness_ratio).
    mode in {"no_image","edge","ridge","fail"}.
    """
    vr = vertical_brightness_ratio(img)
    if vr == 0.0:
        return None, "no_image", vr

    try:
        img_med = median(img, disk(params.med_disk))
        mode = "edge" if vr > params.ratio_cutoff else "ridge"

        if mode == "edge":
            path, bw = extract_boundary(img_med, threshold_li)
            # sanity fallbacks
            H, W = img.shape
            if path.sum() > 1.5 * W:
                path, bw = extract_boundary(img_med, threshold_minimum, min_size=0, hole_area=0, opening_size=0)
            elif path[0, :].any() or path[-1, :].any():
                g = exposure.adjust_gamma(img_med, gamma=0.2)
                path, bw = extract_boundary(g, threshold_triangle, min_size=0, hole_area=0, opening_size=10)
            path = extend_path_to_edges(path, k=5, max_gap=params.edge_extend_gap)
            return path, "edge", vr

        # ridge mode
        resp = sato(img_med, sigmas=params.sato_sigmas, black_ridges=True)
        resp = exposure.rescale_intensity(resp)
        bw = resp > threshold_otsu(resp)
        bw = binary_closing(bw, footprint=disk(params.ridge_close_disk))
        bw = remove_small_objects(bw, min_size=params.ridge_min_size)

        skel = skeletonize(bw)
        skel_main = _largest_n_components(skel, n_keep=params.ridge_keep_components)

        path = find_best_left_right_route(skel_main, edge_margin=params.edge_margin, step_cost=params.step_cost)
        if path is None:
            path = find_best_left_right_route(skel, edge_margin=params.edge_margin, step_cost=params.step_cost)
        if path is not None:
            path = extend_path_to_edges(path, k=5, max_gap=params.ridge_extend_gap)
        return path, "ridge", vr

    except Exception:
        return None, "fail", vr


# -------------------------
# Profile + z collection
# -------------------------

def pathmask_to_profile_y(pm: np.ndarray) -> Optional[np.ndarray]:
    """Convert a sparse 1px-ish path mask into a dense y(x) profile of length W."""
    H, W = pm.shape
    y, x = np.nonzero(pm)
    if y.size < 2:
        return None
    order = np.argsort(x)
    x = x[order].astype(np.float32)
    y = y[order].astype(np.float32)
    x_u, idx = np.unique(x, return_index=True)
    y_u = y[idx]
    if x_u.size < 2:
        return None
    return np.interp(np.arange(W, dtype=np.float32), x_u, y_u, left=y_u[0], right=y_u[-1]).astype(np.float32)


def collect_lines_over_z(
    vol_zyx: np.ndarray,
    params: RidgeParams = RidgeParams(),
    discard_top_bottom: bool = True,
) -> Tuple[Dict[int, np.ndarray], Dict[int, np.ndarray], List[int]]:
    """
    Returns:
      lines[z]    -> bool (H,W)
      profiles[z] -> float32 (W,)
      found_zs    -> sorted list of z where a line exists
    """
    Z, H, W = vol_zyx.shape
    lines: Dict[int, np.ndarray] = {}
    profiles: Dict[int, np.ndarray] = {}
    found: List[int] = []

    for z in range(Z):
        pm, mode, vr = find_ridge_through_plane(vol_zyx[z], params=params)
        if pm is None or pm.sum() == 0:
            continue
        pm = pm.astype(bool)
        if discard_top_bottom and (pm[0, :].any() or pm[-1, :].any()):
            continue
        yx = pathmask_to_profile_y(pm)
        if yx is None:
            continue
        lines[z] = pm
        profiles[z] = yx
        found.append(z)

    return lines, profiles, sorted(found)


# -------------------------
# Keep consistent z-band
# -------------------------

def keep_similar_high_z(
    lines: Dict[int, np.ndarray],
    profiles: Dict[int, np.ndarray],
    found_zs: List[int],
    *,
    missing_run: int = 3,
    diff_thresh_px: Optional[float] = None,
    core_frac: float = 0.6,
    core_min: int = 5,
    min_keep: int = 3,
    consec_bad: int = 2,
) -> Tuple[List[int], Dict[int, np.ndarray]]:
    """
    Keep the main contiguous z-band (trims weird ends), with an initial low-z cutoff before a missing-run.

    - Pre-cut: if there is a gap implying >= missing_run absent slices, drop all z <= a+missing_run.
    - Consensus: median profile of a central core band.
    - Keep: best contiguous block of slices with diff<=thr (allows up to consec_bad-1 consecutive failures).
    """
    zs = sorted([z for z in found_zs if z in lines and z in profiles])
    if not zs:
        return [], {}

    if len(zs) == 1:
        z0 = zs[0]
        return [z0], {z0: lines[z0]}

    # pre-cut low-z before first missing run
    cut_z = None
    for a, b in zip(zs[:-1], zs[1:]):
        if (b - a - 1) >= missing_run:
            cut_z = a + missing_run
            break
    if cut_z is not None:
        zs = [z for z in zs if z > cut_z]
    if not zs:
        return [], {}

    n = len(zs)
    if n == 1:
        z0 = zs[0]
        return [z0], {z0: lines[z0]}

    core_n = min(n, max(core_min, int(np.ceil(core_frac * n))))
    start = (n - core_n) // 2
    core = zs[start : start + core_n]
    consensus = np.median(np.stack([profiles[z] for z in core], axis=0), axis=0)

    diffs = np.array([np.median(np.abs(profiles[z] - consensus)) for z in zs], dtype=np.float32)

    if diff_thresh_px is None:
        core_diffs = np.array([np.median(np.abs(profiles[z] - consensus)) for z in core], dtype=np.float32)
        med = float(np.median(core_diffs))
        mad = float(np.median(np.abs(core_diffs - med))) + 1e-9
        thr = med + 3.0 * 1.4826 * mad
    else:
        thr = float(diff_thresh_px)

    good = diffs <= thr

    best: Optional[Tuple[Tuple[int, int], int, int]] = None  # (score, i0, i1)
    i = 0
    while i < n:
        if not good[i]:
            i += 1
            continue

        i0, bad_run, j = i, 0, i
        while j < n:
            if good[j]:
                bad_run = 0
            else:
                bad_run += 1
                if bad_run >= consec_bad:
                    break
            j += 1

        i1 = j
        score = (int(good[i0:i1].sum()), i1 - i0)
        if best is None or score > best[0]:
            best = (score, i0, i1)
        i = i1

    if best is None:
        return [], {}

    _, i0, i1 = best
    kept = [z for z, g in zip(zs[i0:i1], good[i0:i1]) if bool(g)]

    if len(kept) < min_keep:
        kept = [zs[int(np.argmin(diffs))]]

    return kept, {z: lines[z] for z in kept}


# -------------------------
# Surface smoothing + segmentation
# -------------------------

def smooth_surface_yzx(
    profiles: Dict[int, np.ndarray],
    kept_zs: List[int],
    Z: int,
    W: int,
    *,
    sigma_z: float = 1.0,
    sigma_x: float = 2.0,
) -> np.ndarray:
    """Build y[z,x] from kept_zs, interpolate along z per x, then smooth in (z,x)."""
    ysurf = np.full((Z, W), np.nan, dtype=np.float32)
    for z in kept_zs:
        ysurf[z] = profiles[z].astype(np.float32, copy=False)

    zz = np.arange(Z, dtype=np.float32)
    for x in range(W):
        col = ysurf[:, x]
        ok = ~np.isnan(col)
        if ok.sum() >= 2:
            ysurf[:, x] = np.interp(zz, zz[ok], col[ok]).astype(np.float32)
        elif ok.sum() == 1:
            ysurf[:, x] = float(col[ok][0])

    return gaussian_filter(ysurf, sigma=(sigma_z, sigma_x), mode="nearest")


def segment_from_smoothed_surface(
    vol_shape_zyx: Tuple[int, int, int],
    kept_zs: List[int],
    profiles: Dict[int, np.ndarray],
    *,
    side: str = "below",
    sigma_z: float = 1.0,
    sigma_x: float = 2.0,
) -> Tuple[np.ndarray, np.ndarray, Dict[int, int], int, int, np.ndarray]:
    """
    Segment volume into region/other using a smoothed boundary surface y(z,x).
    Returns (region, other, area_by_z, vol_region, vol_other, y_smooth_zx).
    """
    Z, H, W = vol_shape_zyx
    y_smooth = smooth_surface_yzx(profiles, kept_zs, Z, W, sigma_z=sigma_z, sigma_x=sigma_x)

    region = np.zeros((Z, H, W), dtype=bool)
    other = np.zeros((Z, H, W), dtype=bool)
    area_by_z: Dict[int, int] = {}

    Y = np.arange(H)[:, None]
    for z in kept_zs:
        yy = np.clip(np.rint(y_smooth[z]).astype(np.int32), 0, H - 1)[None, :]
        below = (Y >= yy)
        if side == "below":
            region[z] = below
            other[z] = ~below
            area_by_z[z] = int(below.sum())
        elif side == "above":
            region[z] = ~below
            other[z] = below
            area_by_z[z] = int((~below).sum())
        else:
            raise ValueError("side must be 'below' or 'above'")

    return region, other, area_by_z, int(region.sum()), int(other.sum()), y_smooth

In [ ]:

# def vertical_brightness_ratio(img, eps=1e-6):
#     """
#     Measures brightness discontinuity top vs bottom.
#     Returns absolute log ratio: abs(log(top/bottom)).
#     """
#     y = int(img.shape[0] * 0.5)
#     a = np.median(img[:y])
#     b = np.median(img[y:])
#     return float(abs(np.log((a + eps) / (b + eps))))

# def fit_line_fast(x, y):
#     """
#     Fast least-squares line fit y = a*x + b.
#     Much faster than np.polyfit for small k.
#     """
#     n = x.size
#     if n < 2:
#         return 0.0, float(y[0]) if n else 0.0

#     x = x.astype(np.float32, copy=False)
#     y = y.astype(np.float32, copy=False)

#     xm = x.mean()
#     ym = y.mean()

#     dx = x - xm
#     denom = np.dot(dx, dx)

#     if denom == 0:
#         return 0.0, float(ym)

#     a = np.dot(dx, y - ym) / denom
#     b = ym - a * xm
#     return float(a), float(b)

# def extend_path_to_edges(path_mask, k=5, max_gap=30, edge_tol=0):
#     """
#     Extend a path so it touches x=0 and x=W-1, but only if endpoints are within max_gap.
#     Optimised for speed: argpartition + closed-form line fit.
#     """
#     m = path_mask.astype(bool).copy()
#     H, W = m.shape

#     ys, xs = np.nonzero(m)
#     n = xs.size
#     if n < 2:
#         return m

#     x_min = int(xs.min())
#     x_max = int(xs.max())
#     left_gap  = x_min
#     right_gap = (W - 1) - x_max

#     if left_gap <= edge_tol and right_gap <= edge_tol:
#         return m

#     if left_gap > max_gap and right_gap > max_gap:
#         return m

#     kk = min(k, n)

#     # ---- LEFT EXTENSION ----
#     if left_gap <= max_gap:
#         idx = np.argpartition(xs, kk - 1)[:kk]
#         a, b = fit_line_fast(xs[idx], ys[idx])

#         x0 = x_min
#         y0 = int(np.clip(np.rint(a * x0 + b), 0, H - 1))
#         y_edge = int(np.clip(np.rint(b), 0, H - 1))

#         rr, cc = sk_line(y_edge, 0, y0, x0)
#         m[rr, cc] = True

#     # ---- RIGHT EXTENSION ----
#     if right_gap <= max_gap:
#         idx = np.argpartition(xs, n - kk)[-kk:]
#         a, b = fit_line_fast(xs[idx], ys[idx])

#         x1 = x_max
#         y1 = int(np.clip(np.rint(a * x1 + b), 0, H - 1))
#         y_edge = int(np.clip(np.rint(a * (W - 1) + b), 0, H - 1))

#         rr, cc = sk_line(y1, x1, y_edge, W - 1)
#         m[rr, cc] = True

#     return m


# def extract_boundary(img, thr_func, min_size=100, area_threshold=500, opening_size=3): 
#     bw_orig = img > thr_func(img)
#     bw = remove_small_objects(bw_orig, min_size=min_size)
#     bw = remove_small_holes(bw, area_threshold=area_threshold)
#     bw = binary_opening(bw, disk(opening_size))


#     labels = label(bw)
#     if labels.max() > 0:
#         props = regionprops(labels)
#         biggest = max(props, key=lambda p: p.area)
#         bw = (labels == biggest.label)

#     # interface line
#     path_mask = bw ^ binary_erosion(bw, disk(1))
#     lab = label(path_mask)
#     if lab.max() > 0:
#         props = regionprops(lab)
#         longest = max(props, key=lambda p: p.area) # area == number of pixels
#         path_mask = (lab == longest.label)
#     return path_mask, bw

# from heapq import heappush, heappop

# def find_best_left_right_route(mask, edge_margin=30, step_cost=100.0):
#     H, W = mask.shape

#     left_pts  = np.argwhere(mask[:, :edge_margin])
#     right_pts = np.argwhere(mask[:, W-edge_margin:])

#     if len(left_pts) == 0 or len(right_pts) == 0:
#         m2 = binary_dilation(mask, disk(2))
#         left_pts  = np.argwhere(m2[:, :edge_margin])
#         right_pts = np.argwhere(m2[:, W-edge_margin:])
#         if len(left_pts) == 0 or len(right_pts) == 0:
#             return None
#         mask = m2

#     right_set = {(y, W-edge_margin + x) for y, x in right_pts}

#     off_penalty = step_cost * 0.2
#     cost = np.where(mask, step_cost, step_cost + off_penalty)

#     visited = np.full(mask.shape, False, dtype=bool)
#     prev = dict()
#     pq = []

#     # Multi-source init
#     for y, x in left_pts:
#         heappush(pq, (0.0, (y, x)))
#         prev[(y, x)] = None

#     end = None

#     while pq:
#         c, (y, x) = heappop(pq)
#         if visited[y, x]:
#             continue
#         visited[y, x] = True

#         if (y, x) in right_set:
#             end = (y, x)
#             break

#         for dy, dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
#             ny, nx = y+dy, x+dx
#             if 0 <= ny < H and 0 <= nx < W and not visited[ny, nx]:
#                 nc = c + cost[ny, nx]
#                 heappush(pq, (nc, (ny, nx)))
#                 if (ny, nx) not in prev:
#                     prev[(ny, nx)] = (y, x)

#     if end is None:
#         return None

#     # Reconstruct path
#     path_mask = np.zeros_like(mask, dtype=bool)
#     cur = end
#     while cur is not None:
#         path_mask[cur] = True
#         cur = prev[cur]

#     return path_mask



# def find_ridge_through_plane(img_slice, to_plot=True, ratio_cutoff=0.75):
#     # # Gate: decide edge-mode vs ridge-mode
#     vr = vertical_brightness_ratio(img_slice)
#     try:
    
#         if vr == 0:
#             mode = "no image"
#             title = "skipping image"
#             path_mask=None
#             if to_plot==True:
#                 bw = np.zeros_like(img_slice)
            
#         else:
#             mode = "edge" if vr > ratio_cutoff else "ridge"    
#             if mode == "edge": 
#                 H, W = img_slice.shape
#                 # --- First attempt: Li ---
#                 img_med = median(img_slice, disk(7))
#                 path_mask, bw = extract_boundary(img_med, threshold_li)
#                 title = "li"
                
#                 # --- Sanity check: boundary too long? ---
#                 if path_mask.sum() > 1.5 * W:
#                     title = "minimum"
#                     path_mask, bw = extract_boundary(img_med, threshold_minimum, min_size = 0, area_threshold = 0, opening_size=0)
#                 elif path_mask[0, :].any() or path_mask[-1, :].any():
#                     title = "triangle"
#                     path_mask, bw = extract_boundary(exposure.adjust_gamma(img_med, gamma=0.2), threshold_triangle, min_size = 0, area_threshold = 0, opening_size=10)
                    
#                 path_mask = extend_path_to_edges(path_mask,  k=5, max_gap=30, edge_tol=0)
                        
#             else: ####### ridge mode 
#                 title="ridge"
#                 img_med = median(img_slice, disk(7))

#                 resp = sato(img_med, sigmas=np.linspace(2, 8, 10), black_ridges=True)
#                 resp = exposure.rescale_intensity(resp)
#                 # if vr>0.4:
#                 #     thr = threshold_yen(resp)
#                 # else:
#                 thr = threshold_otsu(resp)
#                 bw = resp > thr

#                 bw = binary_closing(bw, footprint=disk(9))
#                 bw = remove_small_objects(bw, min_size=600)
#                 skel = skeletonize(bw)
#                 lab = label(skel)
#                 if lab.max() > 0:
#                     props = regionprops(lab)
#                     props_sorted = sorted(props, key=lambda p: p.area, reverse=True)
#                     labels_to_keep = [p.label for p in props_sorted[:3]]
#                     skel_main = np.isin(lab, labels_to_keep)
#                 else:
#                     skel_main = skel

#                 path_mask = find_best_left_right_route(skel_main)
                
#                 if path_mask is None:
#                     path_mask = find_best_left_right_route(skel)
#                 if path_mask is not None:
#                     print("now extending on ridge")
#                     path_mask = extend_path_to_edges(path_mask,  k=5, max_gap=50, edge_tol=0)
#     except:
#         mode = "can't segment"
#         title = "skipping image"
#         path_mask=None
#         if to_plot==True:
#             bw = np.zeros_like(img_slice)

#     if to_plot:
#         fig, ax = plt.subplots(ncols=3, figsize=(6, 3))
#         ax[0].imshow(img_slice, cmap="gray")
#         ax[1].imshow(bw, cmap="gray")
#         ax[2].imshow(img_slice, cmap="gray")
#         ax[0].set_title("input")
#         ax[1].set_title(title)
#         ax[2].set_title("path")
#         if path_mask is not None:
#             y, x = np.nonzero(path_mask)
#             ax[2].scatter(x, y, s=2, c="red")
#         for a in ax:
#             a.axis("off")
#         plt.suptitle(f"Mode:{mode}, Brightness ratio:{vr}")
#         plt.tight_layout()
#         plt.show()
#     print(f"Mode={mode}, threshold={title}")
#     return path_mask, mode, vr

# # ----------------------------
# # Helper: line -> y(x) profile
# # ----------------------------
# def _pathmask_to_profile_y(pm):
#     H, W = pm.shape
#     y, x = np.nonzero(pm)
#     if y.size < 2:
#         return None
#     order = np.argsort(x)
#     x = x[order].astype(float)
#     y = y[order].astype(float)
#     x_u, idx = np.unique(x, return_index=True)
#     y_u = y[idx]
#     if x_u.size < 2:
#         return None
#     x_grid = np.arange(W, dtype=float)
#     return np.interp(x_grid, x_u, y_u, left=y_u[0], right=y_u[-1])


# # ============================================================
# # 1) Iterate through z -> return lines and zs where a line exists
# # ============================================================
# def collect_lines_over_z(vol_zyx, ratio_cutoff=0.75, discard_top_bottom=True, to_plot=True):
#     """
#     Returns:
#       lines: dict[z] -> bool path_mask
#       profiles: dict[z] -> float y(x) profile (len W)
#       found_zs: sorted list of z where a valid line exists
#     """
#     Z, H, W = vol_zyx.shape
#     lines = {}
#     profiles = {}
#     found_zs = []

#     for z in range(Z):
#         print(f"slice {z}")
#         pm, mode, vr = find_ridge_through_plane(vol_zyx[z], to_plot=to_plot, ratio_cutoff=ratio_cutoff)
#         if pm is None or pm.sum() == 0:
#             continue
#         pm = pm.astype(bool)
#         if discard_top_bottom and (pm[0, :].any() or pm[-1, :].any()):
#             continue

#         yx = _pathmask_to_profile_y(pm)
#         if yx is None:
#             continue

#         lines[z] = pm
#         profiles[z] = yx
#         found_zs.append(z)

#     return lines, profiles, sorted(found_zs)


# def keep_similar_high_z(
#     lines,
#     profiles,
#     found_zs,
#     # --- NEW: pre-cut based on missing run ---
#     missing_run=3,         # run length of absent z's to trigger low-z cutoff
#     # --- Main trimming params ---
#     diff_thresh_px=None,   # if None: auto from core
#     core_frac=0.6,
#     core_min=5,
#     min_keep=3,
#     consec_bad=2           # allow up to consec_bad-1 consecutive "bad" within a kept block
# ):
#     """
#     0) Pre-cut low-z:
#        If there is a run of `missing_run` consecutive missing slices before the main region,
#        remove all found_zs <= end_of_run (so they don't influence the core/consensus).

#     1) Build a 'main' consensus from a central core band of the remaining found_zs.
#     2) Compute diff(z) to consensus for each remaining slice.
#     3) Keep the best contiguous block (cuts odd slices at top or bottom),
#        allowing small internal bad runs up to `consec_bad-1`.
#     """
#     if not found_zs:
#         return [], {}

#     found_zs = sorted([z for z in found_zs if z in profiles and z in lines])
#     if len(found_zs) == 0:
#         return [], {}

#     # ------------------------------------------------------------
#     # 0) PRE-CUT: remove low-z band before a run of "no path" slices
#     # ------------------------------------------------------------
#     # A run of `missing_run` consecutive missing slices is implied by a gap:
#     #   gap_missing = (b - a - 1) >= missing_run
#     # The end of the first run of length `missing_run` is: cut_z = a + missing_run
#     cut_z = None
#     for a, b in zip(found_zs[:-1], found_zs[1:]):
#         if (b - a - 1) >= missing_run:
#             cut_z = a + missing_run
#             break

#     if cut_z is not None:
#         found_zs = [z for z in found_zs if z > cut_z]

#     if len(found_zs) == 0:
#         return [], {}

#     # -------------------------
#     # 1) define "main" core band
#     # -------------------------
#     n = len(found_zs)
#     if n == 1:
#         z0 = found_zs[0]
#         return [z0], {z0: lines[z0]}

#     core_n = max(core_min, int(np.ceil(core_frac * n)))
#     core_n = min(core_n, n)
#     start = (n - core_n) // 2
#     core_zs = found_zs[start:start + core_n]

#     consensus = np.median(np.stack([profiles[z] for z in core_zs], axis=0), axis=0)

#     # -------------------------
#     # 2) diffs for all slices
#     # -------------------------
#     diffs = np.array([np.median(np.abs(profiles[z] - consensus)) for z in found_zs], dtype=float)

#     # -------------------------
#     # 3) threshold
#     # -------------------------
#     if diff_thresh_px is None:
#         core_diffs = np.array([np.median(np.abs(profiles[z] - consensus)) for z in core_zs], dtype=float)
#         med = float(np.median(core_diffs))
#         mad = float(np.median(np.abs(core_diffs - med))) + 1e-9
#         thr = med + 3.0 * 1.4826 * mad
#     else:
#         thr = float(diff_thresh_px)

#     good = diffs <= thr

#     # ----------------------------------------------------
#     # 4) best contiguous block (trim top/bottom outliers)
#     # ----------------------------------------------------
#     best = None  # (score_tuple, i0, i1)
#     i = 0
#     while i < n:
#         if not good[i]:
#             i += 1
#             continue

#         i0 = i
#         bad_run = 0
#         j = i

#         while j < n:
#             if good[j]:
#                 bad_run = 0
#             else:
#                 bad_run += 1
#                 if bad_run >= consec_bad:
#                     break
#             j += 1

#         i1 = j  # [i0, i1)
#         block_len = i1 - i0
#         block_good = int(good[i0:i1].sum())
#         score = (block_good, block_len)  # prioritize more good slices, then longer block

#         if best is None or score > best[0]:
#             best = (score, i0, i1)

#         i = i1

#     if best is None:
#         return [], {}

#     _, i0, i1 = best
#     kept_zs = [z for z, g in zip(found_zs[i0:i1], good[i0:i1]) if g]

#     if len(kept_zs) < min_keep:
#         # fallback: keep the single closest slice to consensus
#         z_best = found_zs[int(np.argmin(diffs))]
#         kept_zs = [z_best]

#     kept_lines = {z: lines[z] for z in kept_zs}
#     return kept_zs, kept_lines
# # ============================================================
# # 3) Build segmentations from kept lines + compute areas/volumes
# # ============================================================
# def segment_from_lines_and_measure(vol_shape_zyx, kept_zs, profiles, side="below"):
#     """
#     Builds two 3D masks over ONLY kept_zs:
#       region_zyx: pixels on chosen side of barrier (above/below)
#       other_zyx:  opposite side
#     Also returns:
#       area_by_z: cross-sectional area (pixels) on chosen side for each kept z
#       volume_side / volume_other: voxel counts over kept_zs only
#     """
#     Z, H, W = vol_shape_zyx
#     region = np.zeros((Z, H, W), dtype=bool)
#     other = np.zeros((Z, H, W), dtype=bool)
#     area_by_z = {}

#     Y = np.arange(H)[:, None]  # (H,1)

#     for z in kept_zs:
#         yx = profiles.get(z, None)
#         if yx is None:
#             continue
#         yy = np.clip(np.round(yx).astype(int), 0, H - 1)[None, :]  # (1,W)

#         below = (Y >= yy)
#         above = (Y <= yy)

#         if side == "below":
#             region[z] = below
#             other[z] = ~below
#             area_by_z[z] = int(below.sum())
#         elif side == "above":
#             region[z] = above
#             other[z] = ~above
#             area_by_z[z] = int(above.sum())
#         else:
#             raise ValueError("side must be 'below' or 'above'")

#     volume_side = int(region.sum())
#     volume_other = int(other.sum())  # only nonzero on kept_zs

#     return region, other, area_by_z, volume_side, volume_other

# def lines_dict_to_label_volume(lines, Z, H, W):
#     lab = np.zeros((Z, H, W), dtype=np.uint8)
#     for z, pm in lines.items():
#         lab[z][pm] = 1
#     return lab


# import numpy as np
# from scipy.ndimage import gaussian_filter

# def smooth_surface_yzx(profiles, kept_zs, Z, W, sigma_z=1.0, sigma_x=2.0):
#     """
#     Build dense surface y[z, x] for kept_zs, interpolate missing kept zs if any,
#     then apply 2D Gaussian smoothing over (z, x).
#     """
#     ysurf = np.full((Z, W), np.nan, dtype=np.float32)
#     for z in kept_zs:
#         ysurf[z] = profiles[z].astype(np.float32, copy=False)

#     # Fill NaNs along z for each x (linear interpolation)
#     zz = np.arange(Z)
#     for x in range(W):
#         col = ysurf[:, x]
#         ok = ~np.isnan(col)
#         if ok.sum() >= 2:
#             ysurf[:, x] = np.interp(zz, zz[ok], col[ok])
#         elif ok.sum() == 1:
#             ysurf[:, x] = col[ok][0]

#     # Smooth in (z, x). Keep sigma_z small (don’t smear too much across planes).
#     ysmooth = gaussian_filter(ysurf, sigma=(sigma_z, sigma_x), mode="nearest")
#     return ysmooth

# def segment_from_smoothed_surface(vol_shape_zyx, kept_zs, profiles, side="below",
#                                   sigma_z=1.0, sigma_x=2.0):
#     Z, H, W = vol_shape_zyx
#     ysmooth = smooth_surface_yzx(profiles, kept_zs, Z, W, sigma_z=sigma_z, sigma_x=sigma_x)

#     region = np.zeros((Z, H, W), dtype=bool)
#     other  = np.zeros((Z, H, W), dtype=bool)
#     area_by_z = {}

#     Y = np.arange(H)[:, None]  # (H,1)

#     for z in kept_zs:
#         yy = np.clip(np.rint(ysmooth[z]).astype(np.int32), 0, H - 1)[None, :]  # (1,W)

#         below = (Y >= yy)
#         if side == "below":
#             region[z] = below
#             other[z]  = ~below
#             area_by_z[z] = int(below.sum())
#         elif side == "above":
#             region[z] = ~below
#             other[z]  = below
#             area_by_z[z] = int((~below).sum())
#         else:
#             raise ValueError("side must be 'below' or 'above'")

#     volume_side  = int(region.sum())
#     volume_other = int(other.sum())
#     return region, other, area_by_z, volume_side, volume_other, ysmooth

In [517]:
folder = Path(r"Z:\Maria Cuende\0_Projects\0_Placenta\Barrier integrity\20260115-22_Exp6-Thalidomide-Ibuprofen")
lif_files = list(folder.glob("*.lif"))
print(len(lif_files))

lif_path = lif_files[1]
image_index = 1


dextran_channel = 2
with LifFile(lif_path) as lif:
    img = lif.images[image_index]
    print(f"Opening image {lif.name}, which has dimensions {getattr(img, 'dims', None)} and shape {img.shape}")
    img = img.asarray()
dextran_t0 = img[0, dextran_channel, :, :, :]
print("dextran_t0 shape:", dextran_t0.shape)


    # for z in [5,7,12,15,18,20,25]:
    #     path_mask, mode, vr = find_ridge_through_plane(dextran_t0[z, :, :], to_plot=True)

2
Opening image Exp6_20260122_Thalidomide-Ibuprofen.lif, which has dimensions ('T', 'C', 'Z', 'Y', 'X') and shape (4, 3, 41, 512, 512)
dextran_t0 shape: (41, 512, 512)


In [518]:
# dextran_t0 is (Z, H, W)
params = RidgeParams(
    ratio_cutoff=0.9,
    # optional speed tweak:
    sato_sigmas=(2, 3, 4, 5, 6, 7, 8),
)

# 1) detect per-slice lines + profiles
lines, profiles, found_zs = collect_lines_over_z(
    dextran_t0,
    params=params,
    discard_top_bottom=True,
)

# 2) keep consistent z-band
kept_zs, kept_lines = keep_similar_high_z(
    lines,
    profiles,
    found_zs,
    missing_run=3,
    consec_bad=2,
    core_frac=0.6,
)

# 3) segment using a smoothed surface (prevents jagged overlap artefacts)
region_zyx, other_zyx, area_by_z, vol_region, vol_other, y_smooth_zx = segment_from_smoothed_surface(
    dextran_t0.shape,
    kept_zs,
    profiles,
    side="below",   # or "above"
    sigma_z=1.0,
    sigma_x=3.0,
)

print("found_zs:", len(found_zs), "kept_zs:", len(kept_zs))
print("vol_region:", vol_region, "vol_other:", vol_other)

found_zs: 36 kept_zs: 27
vol_region: 3902135 vol_other: 3175753


In [520]:
viewer = napari.Viewer()
viewer.add_image(dextran_t0)
viewer.add_labels(region_zyx , colormap=label_colormap("dodgerblue", alpha=0.2))
viewer.add_labels(other_zyx, colormap={0: np.array([0., 0., 0., 0.]), 1:np.array([1.,0.,0.,0.2])} )



<Labels layer 'other_zyx' at 0x160553fa0d0>